# Flow Past a Spinning Obstacle (2D)

A rigid body sits in a periodic box of fluid, **spinning** at `obstacleOmega`,
while the flow around it is driven towards `U_target`. The wake is the point:
the body sheds vortices, the spin makes the shedding asymmetric (a Magnus-like
bias towards one side), and because the box is periodic the wake eventually
comes back around and interacts with the body again.

Two mechanisms are worth reading before changing numbers:

- **The forcing acts on the mean velocity only.** `meanFlowForcing` measures
  the domain-mean fluid velocity and applies
  `m (U_target - mean) / forcingTau` to every fluid particle -- so it corrects
  the *average* towards the target on a timescale `forcingTau` and leaves the
  fluctuations alone. Forcing each particle towards `U_target` individually
  would damp exactly the wake this case exists to show.
- **The obstacle is a rigid body, not a wall.** It is a boundary region with
  `BCType.constant`, which the initializer turns into a rigid body;
  `initialConditions` then sets `angularVelocity` on it. That is why the spin
  is a case parameter rather than a moving-region trick.

The obstacle shape is a parameter (`--obstacleShape`, any key of
`SHAPE_PRESETS`), as are its aspect, its rotation and its position, so "wake
behind a spinning hexagon" and "wake behind a spinning star" are the same run
twice. A `circle` is the null experiment: a spinning circle presents the same
outline at every instant, so the shedding it produces comes only from the
no-slip drag.

![](outputs/10-movingObstacle.gif)


## Every knob, and what it does

The parameters cell below is the whole command line of `10-moving-obstacle.py` written out:
`CaseSpec` fields first, then `movingObstacleCase.params` -- the case's own physics knobs,
each of which is also a `--flag`. Anything not named there keeps the value in
`movingObstacleCase.defaults`/`.params`.

**Discretisation, time stepping and output** (`CaseSpec` fields, shared by every case)

| field | this notebook | what it does |
|---|---|---|
| `nx` | `128` | particles across the domain; the spacing is `dx = L / nx` |
| `dim` | `2` | this case is 2D |
| `L` | `2.0` | side of the (periodic) box |
| `n_h` | `4.0` | particles per support radius, i.e. how smooth the kernel is |
| `kernel` | `Wendland4` | SPH kernel |
| `integrationScheme` | `rungeKutta2` | time integrator |
| `scheme` | `deltaSPH` | the solver itself |
| `tLimit` | `10.0` | simulated end time; the loop runs `tLimit / dt` steps |
| `dt` | *set by the case* | left `None`: `initialConditions` picks it together with the sound speed |
| `adaptiveDt`, `cflFactor`, `minDt` | `True`, `0.3`, `1e-8` | CFL limiter around that `dt` |
| `plot`, `show`, `plotInterval` | `True`, `True`, `10` | render a frame every `plotInterval` steps |
| `store`, `storeMode`, `storeInterval` | `False`, `'states'`, `500` | HDF5 export; off here |

**The case's own parameters** (`--flag` on the script, `params=dict(...)` here)

| parameter | this notebook | what it does |
|---|---|---|
| `obstacleShape` | `'hexagon'` | any key of `SHAPE_PRESETS`: `circle`, `box`, `roundedBox`, `rhombus`, `trapezoid`, `parallelogram`, `equilateralTriangle`, `triangleIsosceles`, `pentagon`, `hexagon`, `octogon`, `hexagram`, `star5`, `vesica`, `cutDisk`, `unevenCapsule`, `moon` |
| `obstacleSize` | `0.25` | characteristic half-size of the body |
| `obstacleAspect` | `1.0` | squashes it in its second direction |
| `obstacleRotation` | `0.0` | degrees counter-clockwise, its initial orientation |
| `obstacleOffset` | `[0.0, 0.0]` | where its (measured) centre sits; a list, so `--config`/notebook only |
| `obstacleOmega` | `1.0` | rad/s the body spins at, counter-clockwise |
| `U_target` | `1.0` | the mean x-velocity the forcing drives the fluid towards |
| `forcingTau` | `0.5` | timescale of that forcing; larger is gentler |
| `rho0` | `1.0` | rest density |
| `targetDt` | `0.0005` | the timestep the run *asks* for; the sound speed is then chosen to make it the acoustic CFL limit |
| `inviscid`, `nu` | `True`, `0.0` | physical viscosity: `inviscid=True` leaves the scheme's own dissipation as the only one |
| `freeSurface` | `False` | surface detection, on for a case with a free surface |
| `band` | `0` | particle layers of boundary padding around the domain |
| `markerSize` | `8` | plot only: particle marker size |


**Three things this family does differently from the compressible notebooks**
(they will bite if `../compressible/08-Hydrostatic.ipynb` is copied unread):

1. The IC cell has a **fourth call**, `movingObstacleCase.initialConditions(ctx, system)`.
   That is where `setupWeaklyCompressibleTimestep` picks the sound speed and
   `config.dt` *together* from `targetDt` -- weakly compressible SPH is free to
   choose its own stiffness, so the timestep is fixed first and `c0` follows
   from the acoustic CFL. Skip it and `config.dt` stays `None`, and neither the body's spin nor the mean-flow forcing is installed -- both live in `initialConditions`.
2. **The loop is `range(nSteps)`.** No case in this family has a `timestep`
   hook, so `dt` is fixed for the whole run after step 1 and `while t < tLimit`
   would be the wrong shape.
3. Plotting calls `buildFieldPlotter`/`refreshFieldPlotter` on `VELOCITY_DENSITY_FIELDS`
   directly rather than `movingObstacleCase.setupPlot`/`updatePlot`, which go through
   `openWindow`/`pumpEvents` and do not live-update inside a Jupyter cell in
   this environment -- `08-Hydrostatic.ipynb` explains that in full.

Precision note: switching between single and double precision is controlled in
the import cell below. Because precision is set when core modules/kernels are
initialized, any precision change requires a kernel restart.

In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.movingObstacle import movingObstacleCase
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame
from warpSPH.cases.weaklyCompressible import VELOCITY_DENSITY_FIELDS

import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `10-moving-obstacle.py`, made explicit and editable here -- the table in the
# intro cell says what each one does. `movingObstacleCase.defaults`/`.params` are
# the same values the CLI script starts from.
spec = CaseSpec(caseName=movingObstacleCase.name, scheme=movingObstacleCase.scheme,
                params=dict(movingObstacleCase.params)) \
    .merged(**movingObstacleCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=128,
    dim=2,
    L=2.0,

    # --- time stepping ---------------------------------------------------
    tLimit=10.0,

    # --- output --------------------------------------------------------------
    caseName='10-movingObstacle',
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- the obstacle's own knobs ---------------------------------------------
    params=dict(
        # the body: any shape from SHAPE_PRESETS, placed and turned
        obstacleShape='hexagon', obstacleSize=0.25, obstacleAspect=1.0,
        obstacleRotation=0.0, obstacleOffset=[0.0, 0.0],
        # how fast it spins, rad/s
        obstacleOmega=1.0,
        # the driving: mean velocity relaxed towards U_target over forcingTau
        U_target=1.0, forcingTau=0.5,
        # the fluid
        rho0=1.0, targetDt=0.0005, inviscid=True, nu=0.0,
        markerSize=8,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`movingObstacleCase.buildSystem`), not re-derived here.
#
# `initialConditions` is the call the compressible notebooks do not have:
# the body's spin and the mean-flow forcing are installed there, and
# it is where the sound speed and `config.dt` are chosen together from
# `targetDt`, so skipping it leaves `config.dt` unset.
ctx = buildContext(movingObstacleCase, spec)
movingObstacleCase.configureScheme(ctx)
system = movingObstacleCase.buildSystem(ctx)
movingObstacleCase.initialConditions(ctx, system)
runningState = system.initializeNewState()

print(f'dt = {float(ctx.config.dt):.3e}, '
      f'c0 = {ctx.schemeConfig.fluid.fixedSoundSpeed:.3f}, '
      f'{len(runningState.state.positions)} particles')

In [ ]:
# What was actually built: the sampled regions, fluid and boundary, against the
# domain (black) the run is periodic in. This is the cell to look at when a
# geometry parameter above did something other than what it sounded like.
figure, axis = plt.subplots(1, 1, figsize=(5, 5), squeeze=False)
plotRegions(ctx.scratch['regions'], axis[0, 0], plotFluid=True, plotParticles=True)
domain = ctx.config.domain
axis[0, 0].set_aspect('equal')
axis[0, 0].set_xlim(domain.min[0].item(), domain.max[0].item())
axis[0, 0].set_ylim(domain.min[1].item(), domain.max[1].item())
axis[0, 0].set_title(f'{len(ctx.scratch["regions"])} regions, '
                     f'{len(runningState.state.positions)} particles')
figure.tight_layout()

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(VELOCITY_DENSITY_FIELDS), not movingObstacleCase.setupPlot -- see the intro cell
# for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, VELOCITY_DENSITY_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = movingObstacleCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=movingObstacleCase.extraFields)

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
meanVelocity = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    # Injected at the hook point: the domain-mean fluid velocity, which is the
    # quantity the forcing is actually controlling. Nothing else records it.
    particles = runningState.state
    fluid = particles.kinds == 0
    meanVelocity.append(particles.velocities[fluid].mean(dim=0).tolist())
    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = movingObstacleCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, VELOCITY_DENSITY_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=movingObstacleCase.extraFields)

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## Is the forcing doing what it claims?

In [ ]:
# Left: the mean velocity against the target the forcing is driving it to --
# it should approach `U_target` on roughly `forcingTau` and then stay there,
# with the wake showing up as fluctuations around it. Right: the usual density
# bounds.
figure, axis = plt.subplots(1, 2, figsize=(11, 3.5))
t = [row['t'] for row in trajectory]
mean = np.array(meanVelocity)
axis[0].plot(t, mean[:, 0], label=r'$\langle u \rangle$')
axis[0].plot(t, mean[:, 1], label=r'$\langle v \rangle$')
axis[0].axhline(spec.param('U_target'), color='black', ls=':', lw=0.8,
                label='U_target')
axis[0].set_xlabel('t'); axis[0].set_ylabel('mean velocity'); axis[0].legend()
axis[1].plot(t, [row['maxDensity'] for row in trajectory], label='max')
axis[1].plot(t, [row['minDensity'] for row in trajectory], label='min')
axis[1].axhspan(0.99, 1.01, color='green', alpha=0.1, label=r'$\pm 1\%$')
axis[1].set_xlabel('t'); axis[1].set_ylabel(r'$\rho$'); axis[1].legend()
figure.tight_layout()